In [0]:
# CAMADA BRONZE - Ingestao de dados brutos
# Responsavel por: criar o banco bronze e ingerir os CSVs
# do Volume landing_zone sem nenhuma transformacao

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

print("Banco de dados 'bronze' criado com sucesso.")

Banco de dados 'bronze' criado com sucesso.


In [0]:
# Ingestao dos CSVs do Volume para as tabelas Bronze
# Cada CSV e lido e salvo como tabela Delta com a coluna
# timestamp_ingestion registrando o momento exato da carga

from pyspark.sql.functions import current_timestamp

VOLUME_PATH = "/Volumes/workspace/default/landing_zone"

tabelas = {
    "olist_customers_dataset.csv":           "bronze.tb_customers",
    "olist_geolocation_dataset.csv":         "bronze.tb_geolocalizacao",
    "olist_order_items_dataset.csv":         "bronze.tb_order_items",
    "olist_order_payments_dataset.csv":      "bronze.tb_order_payments",
    "olist_order_reviews_dataset.csv":       "bronze.tb_order_reviews",
    "olist_orders_dataset.csv":              "bronze.tb_orders",
    "olist_products_dataset.csv":            "bronze.tb_products",
    "olist_sellers_dataset.csv":             "bronze.tb_sellers",
    "product_category_name_translation.csv": "bronze.tb_product_category_name_translation",
}

for arquivo, tabela in tabelas.items():
    caminho = f"{VOLUME_PATH}/{arquivo}"

    df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(caminho))

    df = df.withColumn("timestamp_ingestion", current_timestamp())

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))

    print(f"Tabela {tabela} carregada com {df.count()} registros.")

print("Ingestao das tabelas Bronze concluida.")

Tabela bronze.tb_customers carregada com 99441 registros.
Tabela bronze.tb_geolocalizacao carregada com 1000163 registros.
Tabela bronze.tb_order_items carregada com 112650 registros.
Tabela bronze.tb_order_payments carregada com 103886 registros.
Tabela bronze.tb_order_reviews carregada com 104162 registros.
Tabela bronze.tb_orders carregada com 99441 registros.
Tabela bronze.tb_products carregada com 32951 registros.
Tabela bronze.tb_sellers carregada com 3095 registros.
Tabela bronze.tb_product_category_name_translation carregada com 71 registros.
Ingestao das tabelas Bronze concluida.


In [0]:
# Ingestao da cotacao do dolar
# A API do Banco Central e inacessivel no ambiente Databricks
# Trial por restricao de rede. Como solucao, o script de coleta
# foi executado localmente e o CSV resultante foi carregado
# no Volume, seguindo o mesmo fluxo dos demais arquivos brutos.

from pyspark.sql.functions import current_timestamp, to_timestamp

df_cotacao = (spark.read
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .csv("/Volumes/workspace/default/landing_zone/cotacao_dolar.csv"))

df_cotacao = df_cotacao.withColumn(
    "dataHoraCotacao", to_timestamp("dataHoraCotacao", "yyyy-MM-dd HH:mm:ss.S")
)

df_cotacao = df_cotacao.withColumn("timestamp_ingestion", current_timestamp())

(df_cotacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.tb_cotacao_dolar"))

print(f"Tabela bronze.tb_cotacao_dolar carregada com {df_cotacao.count()} registros.")

Tabela bronze.tb_cotacao_dolar carregada com 542 registros.
